# GP fit from central values only

This notebook reads only the first centrality block (`0.0-5.0`) from `data/Table01.csv` and uses only the observable central values for the Gaussian Process fit.

The experimental uncertainty columns are intentionally not used in the fit. The goal is to test whether a GP with an RBF correlated component and a white-noise component can infer uncertainty scales comparable to the experimental ones.

Model:

$$\log y_i = f(p_{T,i}) + \epsilon_i,$$

$$f \sim \mathcal{GP}(0, \sigma_f^2 R_{\ell=0.2}), \qquad \epsilon_i \sim \mathcal{N}(0, \sigma_n^2).$$

The fixed correlation length is $\ell=0.2$. The kernel amplitude $\sigma_f$ and white-noise amplitude $\sigma_n$ are estimated from the central values.

In [1]:
from pathlib import Path
import csv

import numpy as np
from IPython.display import HTML, display

try:
    from scipy.optimize import minimize
except ImportError:
    minimize = None

DATA_FILE = Path("data/Table01.csv")
LENGTH_SCALE = 0.2
JITTER = 1e-10

In [2]:
def read_first_centrality_block(path):
    """Read the first HEPData block.

    The GP fit uses only pT and the central observable value. The uncertainty
    columns are kept only for the comparison plot.
    """
    rows = []
    centrality = None
    header = None
    reading_data = False

    with path.open("r", encoding="utf-8", newline="") as f:
        for row in csv.reader(f):
            if not row:
                if reading_data:
                    break
                continue

            first = row[0].strip()
            if first.startswith("#: CENTRALITY") and centrality is None:
                centrality = row[3].strip()
                continue

            if centrality is None:
                continue

            if first.startswith("$p_{T}$"):
                header = row
                reading_data = True
                continue

            if reading_data:
                rows.append([float(value) for value in row])

    if centrality is None or header is None or not rows:
        raise ValueError(f"Could not find the first centrality block in {path}")

    data = np.asarray(rows, dtype=float)
    return centrality, data


centrality, raw = read_first_centrality_block(DATA_FILE)

pT = raw[:, 0]
pT_low = raw[:, 1]
pT_high = raw[:, 2]
y = raw[:, 3]
stat_exp = 0.5 * (np.abs(raw[:, 4]) + np.abs(raw[:, 5]))
syst_exp = 0.5 * (np.abs(raw[:, 6]) + np.abs(raw[:, 7]))
syst_uncorr_exp = 0.5 * (np.abs(raw[:, 8]) + np.abs(raw[:, 9]))

# Experimental components used only for plotting/comparison.
sigma_uncorr_exp = np.sqrt(stat_exp**2 + syst_uncorr_exp**2)
sigma_corr_exp = np.sqrt(np.maximum(syst_exp**2 - syst_uncorr_exp**2, 0.0))

log_y = np.log(y)

print(f"Centrality block: {centrality}%")
print(f"Number of data points: {len(pT)}")
print("The GP fit uses only pT and central observable values.")
print("Experimental uncertainty columns are kept only for the ROOT comparison plot.")

Centrality block: 0.0-5.0%
Number of data points: 58
The GP fit uses only pT and central observable values.
Experimental uncertainty columns are kept only for the ROOT comparison plot.


In [3]:
def rbf_correlation(x1, x2, length_scale=LENGTH_SCALE):
    x1 = np.asarray(x1, dtype=float).reshape(-1, 1)
    x2 = np.asarray(x2, dtype=float).reshape(1, -1)
    return np.exp(-0.5 * ((x1 - x2) / length_scale) ** 2)


def stable_cholesky(K):
    jitter = JITTER
    for _ in range(8):
        try:
            return np.linalg.cholesky(K + jitter * np.eye(K.shape[0]))
        except np.linalg.LinAlgError:
            jitter *= 10.0
    raise np.linalg.LinAlgError("Cholesky decomposition failed even after adding jitter.")


z = log_y - np.mean(log_y)


def negative_log_marginal_likelihood(theta, R_local):
    log_sigma_f, log_sigma_n = theta
    sigma_f = np.exp(log_sigma_f)
    sigma_n = np.exp(log_sigma_n)
    K = sigma_f**2 * R_local + sigma_n**2 * np.eye(len(pT))
    L = stable_cholesky(K)
    alpha = np.linalg.solve(L.T, np.linalg.solve(L, z))
    log_det = 2.0 * np.sum(np.log(np.diag(L)))
    return 0.5 * z @ alpha + 0.5 * log_det + 0.5 * len(pT) * np.log(2.0 * np.pi)


def fit_hyperparameters(R_local):
    initial = np.log([np.std(z), 0.05 * np.std(z)])

    if minimize is not None:
        result = minimize(
            lambda theta: negative_log_marginal_likelihood(theta, R_local),
            initial,
            method="Nelder-Mead",
            options={"maxiter": 5000, "xatol": 1e-10, "fatol": 1e-10},
        )
        if result.success:
            return np.exp(result.x), result.fun, "scipy minimize"

    sigma_f_grid = np.geomspace(0.05 * np.std(z), 5.0 * np.std(z), 80)
    sigma_n_grid = np.geomspace(1e-4 * np.std(z), 1.0 * np.std(z), 80)
    best = None
    for sigma_f in sigma_f_grid:
        for sigma_n in sigma_n_grid:
            value = negative_log_marginal_likelihood(np.log([sigma_f, sigma_n]), R_local)
            if best is None or value < best[0]:
                best = (value, sigma_f, sigma_n)
    return np.array([best[1], best[2]]), best[0], "grid search fallback"


def fit_gp_for_length_scale(length_scale):
    R_local = rbf_correlation(pT, pT, length_scale=length_scale)
    (sigma_f_local, sigma_n_local), nll_local, fit_method_local = fit_hyperparameters(R_local)
    K_corr_local = sigma_f_local**2 * R_local
    K_delta_local = sigma_n_local**2 * np.eye(len(pT))
    K_total_local = K_corr_local + K_delta_local
    return {
        "length_scale": length_scale,
        "R": R_local,
        "sigma_f": sigma_f_local,
        "sigma_n": sigma_n_local,
        "nll": nll_local,
        "fit_method": fit_method_local,
        "K_corr_log": K_corr_local,
        "K_delta_log": K_delta_local,
        "K_total_log": K_total_local,
    }


fit_default = fit_gp_for_length_scale(LENGTH_SCALE)
R = fit_default["R"]
sigma_f = fit_default["sigma_f"]
sigma_n = fit_default["sigma_n"]
nll = fit_default["nll"]
fit_method = fit_default["fit_method"]
K_corr_log = fit_default["K_corr_log"]
K_delta_log = fit_default["K_delta_log"]
K_total_log = fit_default["K_total_log"]
K_corr_log = sigma_f**2 * R
K_delta_log = sigma_n**2 * np.eye(len(pT))
K_total_log = K_corr_log + K_delta_log

print(f"RBF length scale fixed at ell = {LENGTH_SCALE}")
print(f"Fit method: {fit_method}")
print(f"sigma_f on log(y): {sigma_f:.6g}")
print(f"sigma_n on log(y): {sigma_n:.6g}")
print(f"negative log marginal likelihood: {nll:.6g}")

RBF length scale fixed at ell = 0.2
Fit method: scipy minimize
sigma_f on log(y): 4.56018
sigma_n on log(y): 0.0113081
negative log marginal likelihood: 63.02


In [4]:
L = stable_cholesky(K_total_log)
alpha = np.linalg.solve(L.T, np.linalg.solve(L, z))

x_grid = np.linspace(pT.min(), pT.max(), 600)
R_star = rbf_correlation(x_grid, pT)
R_ss_diag = np.ones_like(x_grid)

K_star = sigma_f**2 * R_star
log_mean_grid = np.mean(log_y) + K_star @ alpha

v = np.linalg.solve(L, K_star.T)
log_corr_var_grid = np.maximum(sigma_f**2 * R_ss_diag - np.sum(v**2, axis=0), 0.0)
log_corr_std_grid = np.sqrt(log_corr_var_grid)
log_uncorr_std_grid = np.full_like(x_grid, sigma_n)

R_train = rbf_correlation(pT, pT)
K_train_star = sigma_f**2 * R_train
log_mean_train = np.mean(log_y) + K_train_star @ alpha
v_train = np.linalg.solve(L, K_train_star.T)
log_corr_var_train = np.maximum(sigma_f**2 - np.sum(v_train**2, axis=0), 0.0)
log_corr_std_train = np.sqrt(log_corr_var_train)
log_uncorr_std_train = np.full_like(pT, sigma_n)

gp_mean_grid = np.exp(log_mean_grid)
gp_mean_train = np.exp(log_mean_train)
exp_central_grid = np.exp(np.interp(x_grid, pT, log_y))
exp_corr_band_grid = np.interp(x_grid, pT, sigma_corr_exp)

# Convert log-scale standard deviations to approximate absolute uncertainties.
sigma_corr_model = gp_mean_train * log_corr_std_train
sigma_uncorr_model = gp_mean_train * log_uncorr_std_train
corr_band_grid = gp_mean_grid * log_corr_std_grid
uncorr_band_grid = gp_mean_grid * log_uncorr_std_grid

print(f"K_corr shape: {K_corr_log.shape}")
print(f"K_delta shape: {K_delta_log.shape}")

K_corr shape: (58, 58)
K_delta shape: (58, 58)


In [5]:
def model_uncertainty_table_html(max_rows=None):
    n = len(pT) if max_rows is None else min(max_rows, len(pT))
    rows = []
    for i in range(n):
        rows.append(
            "<tr>"
            f"<td>{i}</td>"
            f"<td>{pT[i]:.6g}</td>"
            f"<td>{y[i]:.6g}</td>"
            f"<td>{gp_mean_train[i]:.6g}</td>"
            f"<td>{gp_mean_train[i] - y[i]:.6g}</td>"
            f"<td>{sigma_uncorr_exp[i]:.6g}</td>"
            f"<td>{sigma_uncorr_model[i]:.6g}</td>"
            f"<td>{sigma_uncorr_model[i] - sigma_uncorr_exp[i]:.6g}</td>"
            f"<td>{sigma_corr_exp[i]:.6g}</td>"
            f"<td>{sigma_corr_model[i]:.6g}</td>"
            f"<td>{sigma_corr_model[i] - sigma_corr_exp[i]:.6g}</td>"
            "</tr>"
        )

    return """
    <table>
      <thead>
        <tr>
          <th>data point</th>
          <th>pT [GeV/c]</th>
          <th>data central value</th>
          <th>GP mean</th>
          <th>diff mean (GP - data)</th>
          <th>exp stat + uncorr syst</th>
          <th>GP uncorrelated uncertainty</th>
          <th>diff uncorrelated (GP - exp)</th>
          <th>exp correlated uncertainty</th>
          <th>GP correlated uncertainty</th>
          <th>diff correlated (GP - exp)</th>
        </tr>
      </thead>
      <tbody>
        {rows}
      </tbody>
    </table>
    """.format(rows="\n".join(rows))


display(HTML(model_uncertainty_table_html()))

data point,pT [GeV/c],data central value,GP mean,diff mean (GP - data),exp stat + uncorr syst,GP uncorrelated uncertainty,diff uncorrelated (GP - exp),exp correlated uncertainty,GP correlated uncertainty,diff correlated (GP - exp)
0,0.11,2049.8,2042.98,-6.8197,41.3612,23.1022,-18.259,140.449,21.2099,-119.239
1,0.13,2187.3,2200.79,13.4856,43.7268,24.8867,-18.8402,103.645,14.5905,-89.0545
2,0.15,2291.6,2299.66,8.0614,45.7113,26.0048,-19.7065,103.267,15.3286,-87.9381
3,0.17,2357.6,2349.66,-7.93621,46.968,26.5702,-20.3978,102.111,14.8625,-87.2482
4,0.19,2384.7,2365.2,-19.4976,47.4732,26.7459,-20.7273,102.53,15.1096,-87.4204
5,0.225,2356.9,2350.2,-6.69527,46.7014,26.5763,-20.1251,102.399,17.5442,-84.8546
6,0.275,2250.6,2307.49,56.8885,44.5992,26.0933,-18.5059,100.729,17.8949,-82.8339
7,0.325,2306,2249.92,-56.081,28.7158,25.4423,-3.27347,47.3135,17.3184,-29.9952
8,0.375,2131.4,2135.8,4.40249,23.0206,24.1518,1.13125,42.1983,16.2091,-25.9892
9,0.425,1937.2,1955.69,18.4926,20.579,22.1151,1.53613,39.8049,14.7222,-25.0827


In [6]:
import uuid
import ROOT


def plot_gp_uncertainty_root_inline(
    x_points, y_points,
    yerr_exp_uncorr, yerr_gp_uncorr,
    x_star, gp_mean_star, gp_corr_sigma_star,
    exp_mean_star, exp_corr_sigma_star,
    *,
    title="Experimental vs GP uncertainty components",
    canvas_name="c_gp_unc",
    width=1100,
    height=560,
):
    ROOT.gROOT.SetBatch(True)
    ROOT.gStyle.SetOptStat(0)

    uid = uuid.uuid4().hex[:6]
    cname = f"{canvas_name}_{uid}"
    c = ROOT.TCanvas(cname, title, width, height)
    c.SetLogy()

    x_points = np.asarray(x_points, dtype=np.float64).ravel()
    y_points = np.asarray(y_points, dtype=np.float64).ravel()
    yerr_exp_uncorr = np.asarray(yerr_exp_uncorr, dtype=np.float64).ravel()
    yerr_gp_uncorr = np.asarray(yerr_gp_uncorr, dtype=np.float64).ravel()
    x_star = np.asarray(x_star, dtype=np.float64).ravel()
    gp_mean_star = np.asarray(gp_mean_star, dtype=np.float64).ravel()
    gp_corr_sigma_star = np.asarray(gp_corr_sigma_star, dtype=np.float64).ravel()
    exp_mean_star = np.asarray(exp_mean_star, dtype=np.float64).ravel()
    exp_corr_sigma_star = np.asarray(exp_corr_sigma_star, dtype=np.float64).ravel()

    zeros_points = np.zeros_like(x_points)
    zeros_star = np.zeros_like(x_star)
    positive_floor = float(np.min(y_points[y_points > 0]) * 1e-3)

    gp_lower = np.maximum(gp_mean_star - gp_corr_sigma_star, positive_floor)
    gp_upper = gp_mean_star + gp_corr_sigma_star
    exp_lower = np.maximum(exp_mean_star - exp_corr_sigma_star, positive_floor)
    exp_upper = exp_mean_star + exp_corr_sigma_star

    ymin = float(min(
        np.min(y_points - yerr_exp_uncorr),
        np.min(y_points - yerr_gp_uncorr),
        np.min(gp_lower),
        np.min(exp_lower),
    ))
    ymin = max(ymin, positive_floor)
    ymax = float(max(
        np.max(y_points + yerr_exp_uncorr),
        np.max(y_points + yerr_gp_uncorr),
        np.max(gp_upper),
        np.max(exp_upper),
    ) * 1.6)
    xmin = float(np.min(x_points))
    xmax = float(np.max(x_points))

    frame = c.DrawFrame(xmin, ymin, xmax, ymax)
    frame.SetTitle(title)
    frame.GetXaxis().SetTitle("p_{T} [GeV/c]")
    frame.GetYaxis().SetTitle("(1/N_{ev}) d^{2}N/(dp_{T} dy)")

    g_exp_band = ROOT.TGraphAsymmErrors(
        len(x_star), x_star, exp_mean_star, zeros_star, zeros_star,
        exp_mean_star - exp_lower, exp_upper - exp_mean_star,
    )
    g_exp_band.SetName(f"g_exp_corr_band_{uid}")
    g_exp_band.SetFillStyle(1001)
    g_exp_band.SetFillColorAlpha(ROOT.kOrange + 7, 0.24)
    g_exp_band.SetLineColorAlpha(ROOT.kOrange + 7, 0.0)

    g_gp_band = ROOT.TGraphAsymmErrors(
        len(x_star), x_star, gp_mean_star, zeros_star, zeros_star,
        gp_mean_star - gp_lower, gp_upper - gp_mean_star,
    )
    g_gp_band.SetName(f"g_gp_corr_band_{uid}")
    g_gp_band.SetFillStyle(1001)
    g_gp_band.SetFillColorAlpha(ROOT.kAzure + 1, 0.26)
    g_gp_band.SetLineColorAlpha(ROOT.kAzure + 1, 0.0)

    g_gp_mean = ROOT.TGraph(len(x_star), x_star, gp_mean_star)
    g_gp_mean.SetName(f"g_gp_mean_{uid}")
    g_gp_mean.SetLineColor(ROOT.kAzure + 2)
    g_gp_mean.SetLineWidth(3)

    g_exp_unc = ROOT.TGraphAsymmErrors(
        len(x_points), x_points, y_points,
        zeros_points, zeros_points, yerr_exp_uncorr, yerr_exp_uncorr,
    )
    g_exp_unc.SetName(f"g_exp_uncorr_{uid}")
    g_exp_unc.SetMarkerStyle(1)
    g_exp_unc.SetMarkerSize(0.01)
    g_exp_unc.SetLineColor(ROOT.kOrange + 7)
    g_exp_unc.SetLineWidth(2)

    g_gp_unc = ROOT.TGraphAsymmErrors(
        len(x_points), x_points, y_points,
        zeros_points, zeros_points, yerr_gp_uncorr, yerr_gp_uncorr,
    )
    g_gp_unc.SetName(f"g_gp_uncorr_{uid}")
    g_gp_unc.SetMarkerStyle(1)
    g_gp_unc.SetMarkerSize(0.01)
    g_gp_unc.SetLineColor(ROOT.kAzure + 2)
    g_gp_unc.SetLineWidth(2)

    g_data = ROOT.TGraph(len(x_points), x_points, y_points)
    g_data.SetName(f"g_data_points_{uid}")
    g_data.SetMarkerStyle(20)
    g_data.SetMarkerSize(0.9)
    g_data.SetMarkerColor(ROOT.kBlack)

    leg = ROOT.TLegend(0.45, 0.60, 0.88, 0.88)
    leg.SetBorderSize(0)
    leg.SetFillStyle(0)
    leg.AddEntry(g_data, "experimental central values", "p")
    leg.AddEntry(g_exp_unc, "exp stat #oplus syst uncorr. uncertainty", "l")
    leg.AddEntry(g_exp_band, "exp correlated syst. uncertainty", "f")
    leg.AddEntry(g_gp_unc, "GP white-noise uncertainty", "l")
    leg.AddEntry(g_gp_band, "GP correlated uncertainty", "f")
    leg.AddEntry(g_gp_mean, "GP mean", "l")

    g_exp_band.Draw("3 SAME")
    g_gp_band.Draw("3 SAME")
    g_gp_mean.Draw("L SAME")
    g_exp_unc.Draw("[] SAME")
    g_gp_unc.Draw("[] SAME")
    g_data.Draw("P SAME")
    leg.Draw()

    c._keep = [frame, g_exp_band, g_gp_band, g_gp_mean, g_exp_unc, g_gp_unc, g_data, leg]
    c.Modified()
    c.Update()

    if hasattr(ROOT, "JSROOT") and hasattr(ROOT.JSROOT, "Draw"):
        return ROOT.JSROOT.Draw(c)
    return c


c_gp_unc = plot_gp_uncertainty_root_inline(
    pT, y,
    sigma_uncorr_exp, sigma_uncorr_model,
    x_grid, gp_mean_grid, corr_band_grid,
    exp_central_grid, exp_corr_band_grid,
    title=f"Experimental and GP uncertainty components: {centrality}%",
)
c_gp_unc

In [7]:
import uuid
import ROOT


def plot_rbf_correlation_matrix_root_inline(R, *, title="Fixed RBF correlation matrix", canvas_name="c_rbf_corr"):
    ROOT.gROOT.SetBatch(True)
    ROOT.gStyle.SetOptStat(0)

    R = np.asarray(R, dtype=np.float64)
    n = R.shape[0]
    uid = uuid.uuid4().hex[:6]
    cname = f"{canvas_name}_{uid}"
    c = ROOT.TCanvas(cname, title, 720, 620)
    c.SetRightMargin(0.14)

    h = ROOT.TH2D(f"h_rbf_corr_{uid}", title, n, -0.5, n - 0.5, n, -0.5, n - 0.5)
    for i in range(n):
        for j in range(n):
            h.SetBinContent(j + 1, i + 1, float(R[i, j]))

    h.SetMinimum(0.0)
    h.SetMaximum(1.0)
    h.GetXaxis().SetTitle("data point j")
    h.GetYaxis().SetTitle("data point i")
    h.GetZaxis().SetTitle("#rho_{ij}")
    h.Draw("COLZ")

    c._keep = [h]
    c.Modified()
    c.Update()

    if hasattr(ROOT, "JSROOT") and hasattr(ROOT.JSROOT, "Draw"):
        return ROOT.JSROOT.Draw(c)
    return c


c_rbf_corr = plot_rbf_correlation_matrix_root_inline(
    R,
    title=f"Fixed RBF correlation matrix, ell = {LENGTH_SCALE}",
)
c_rbf_corr

In [8]:
def gp_uncertainties_at_training_points(fit):
    R_local = fit["R"]
    sigma_f_local = fit["sigma_f"]
    sigma_n_local = fit["sigma_n"]
    K_total_local = fit["K_total_log"]

    L_local = stable_cholesky(K_total_local)
    alpha_local = np.linalg.solve(L_local.T, np.linalg.solve(L_local, z))

    K_train_star_local = sigma_f_local**2 * R_local
    log_mean_train_local = np.mean(log_y) + K_train_star_local @ alpha_local
    v_train_local = np.linalg.solve(L_local, K_train_star_local.T)
    log_corr_var_train_local = np.maximum(
        sigma_f_local**2 - np.sum(v_train_local**2, axis=0), 0.0
    )

    gp_mean_train_local = np.exp(log_mean_train_local)
    sigma_corr_local = gp_mean_train_local * np.sqrt(log_corr_var_train_local)
    sigma_uncorr_local = gp_mean_train_local * sigma_n_local
    return gp_mean_train_local, sigma_corr_local, sigma_uncorr_local


def length_scale_scan_table_html():
    length_scales = np.round(np.arange(0.1, 1.0 + 0.05, 0.1), 1)
    rows = []
    scan_results = []
    pointwise_diffs = {
        "ell": [],
        "mean": [],
        "uncorr": [],
        "corr": [],
    }

    for ell in length_scales:
        fit = fit_gp_for_length_scale(float(ell))
        gp_mean_train_ell, sigma_corr_gp, sigma_uncorr_gp = gp_uncertainties_at_training_points(fit)

        diff_mean = gp_mean_train_ell - y
        diff_uncorr = sigma_uncorr_gp - sigma_uncorr_exp
        diff_corr = sigma_corr_gp - sigma_corr_exp
        pointwise_diffs["ell"].append(float(ell))
        pointwise_diffs["mean"].append(diff_mean)
        pointwise_diffs["uncorr"].append(diff_uncorr)
        pointwise_diffs["corr"].append(diff_corr)

        result = {
            "ell": ell,
            "sigma_f": fit["sigma_f"],
            "sigma_n": fit["sigma_n"],
            "mean_value_diff": np.mean(diff_mean),
            "mean_value_abs_diff": np.mean(np.abs(diff_mean)),
            "mean_value_rms_diff": np.sqrt(np.mean(diff_mean**2)),
            "uncorr_mean_diff": np.mean(diff_uncorr),
            "uncorr_mean_abs_diff": np.mean(np.abs(diff_uncorr)),
            "uncorr_rms_diff": np.sqrt(np.mean(diff_uncorr**2)),
            "corr_mean_diff": np.mean(diff_corr),
            "corr_mean_abs_diff": np.mean(np.abs(diff_corr)),
            "corr_rms_diff": np.sqrt(np.mean(diff_corr**2)),
        }
        scan_results.append(result)
        rows.append(
            "<tr>"
            f"<td>{result['ell']:.1f}</td>"
            f"<td>{result['sigma_f']:.6g}</td>"
            f"<td>{result['sigma_n']:.6g}</td>"
            f"<td>{result['mean_value_diff']:.6g}</td>"
            f"<td>{result['mean_value_abs_diff']:.6g}</td>"
            f"<td>{result['mean_value_rms_diff']:.6g}</td>"
            f"<td>{result['uncorr_mean_diff']:.6g}</td>"
            f"<td>{result['uncorr_mean_abs_diff']:.6g}</td>"
            f"<td>{result['uncorr_rms_diff']:.6g}</td>"
            f"<td>{result['corr_mean_diff']:.6g}</td>"
            f"<td>{result['corr_mean_abs_diff']:.6g}</td>"
            f"<td>{result['corr_rms_diff']:.6g}</td>"
            "</tr>"
        )

    html = """
    <table>
      <thead>
        <tr>
          <th>ell</th>
          <th>sigma_f log(y)</th>
          <th>sigma_n log(y)</th>
          <th>mean diff GP mean (GP - data)</th>
          <th>mean abs diff GP mean</th>
          <th>RMS diff GP mean</th>
          <th>mean diff uncorr (GP - exp)</th>
          <th>mean abs diff uncorr</th>
          <th>RMS diff uncorr</th>
          <th>mean diff corr (GP - exp)</th>
          <th>mean abs diff corr</th>
          <th>RMS diff corr</th>
        </tr>
      </thead>
      <tbody>
        {rows}
      </tbody>
    </table>
    """.format(rows="\n".join(rows))
    pointwise_diffs["ell"] = np.asarray(pointwise_diffs["ell"], dtype=float)
    pointwise_diffs["mean"] = np.asarray(pointwise_diffs["mean"], dtype=float)
    pointwise_diffs["uncorr"] = np.asarray(pointwise_diffs["uncorr"], dtype=float)
    pointwise_diffs["corr"] = np.asarray(pointwise_diffs["corr"], dtype=float)
    return scan_results, pointwise_diffs, html


length_scale_scan, pointwise_diff_scan, length_scale_scan_html = length_scale_scan_table_html()
display(HTML(length_scale_scan_html))

ell,sigma_f log(y),sigma_n log(y),mean diff GP mean (GP - data),mean abs diff GP mean,RMS diff GP mean,mean diff uncorr (GP - exp),mean abs diff uncorr,RMS diff uncorr,mean diff corr (GP - exp),mean abs diff corr,RMS diff corr
0.1,4.26646,0.00576154,-0.00554613,1.23534,3.09793,-6.62443,6.62443,12.1577,-18.2471,18.2471,36.0476
0.2,4.56018,0.0113081,-0.0394712,3.96472,11.4647,-3.16417,3.45798,7.06656,-16.7166,16.7166,33.6108
0.3,4.73727,0.0143729,-0.0763865,6.1752,16.1241,-1.25262,2.83077,4.9855,-16.3086,16.3086,32.803
0.4,4.80583,0.0150705,-0.0862483,7.31783,19.2428,-0.817523,2.71725,4.70006,-16.5732,16.5732,33.0786
0.5,4.6829,0.0153487,-0.0852916,7.97441,20.2777,-0.643979,2.66977,4.60884,-16.8425,16.8425,33.4081
0.6,4.6582,0.0149154,-0.102968,8.12321,20.0651,-0.914521,2.74226,4.75084,-17.2195,17.2195,33.9549
0.7,4.83183,0.014923,-0.151806,8.45344,19.9575,-0.910506,2.74747,4.75735,-17.4047,17.4047,34.2207
0.8,5.14763,0.0158997,-0.244616,9.22903,20.5892,-0.302796,2.59976,4.51742,-17.304,17.304,34.0356
0.9,5.49316,0.0176065,-0.37999,10.6367,22.6171,0.759357,2.38689,4.62699,-17.0172,17.0172,33.552
1.0,5.59016,0.0202255,-0.570445,12.8238,26.4131,2.3886,2.54334,5.92832,-16.531,16.531,32.7445


In [9]:
import uuid
import ROOT


def _draw_zero_line(xmin, xmax):
    line = ROOT.TLine(xmin, 0.0, xmax, 0.0)
    line.SetLineStyle(2)
    line.SetLineColor(ROOT.kGray + 2)
    line.SetLineWidth(1)
    line.Draw("SAME")
    return line


def plot_pointwise_diffs_root_inline(
    x_points,
    diff_mean,
    diff_uncorr,
    diff_corr,
    *,
    title="Pointwise differences for default length scale",
    canvas_name="c_pointwise_diffs",
    width=1050,
    height=900,
):
    ROOT.gROOT.SetBatch(True)
    ROOT.gStyle.SetOptStat(0)

    uid = uuid.uuid4().hex[:6]
    cname = f"{canvas_name}_{uid}"
    c = ROOT.TCanvas(cname, title, width, height)
    c.Divide(1, 3, 0.01, 0.01)

    x_points = np.asarray(x_points, dtype=np.float64).ravel()
    diffs = [
        ("GP mean - data central value", np.asarray(diff_mean, dtype=np.float64), ROOT.kBlack),
        ("GP uncorr. - exp stat #oplus syst uncorr.", np.asarray(diff_uncorr, dtype=np.float64), ROOT.kAzure + 2),
        ("GP corr. - exp corr.", np.asarray(diff_corr, dtype=np.float64), ROOT.kAzure + 2),
    ]

    keep = []
    xmin = float(np.min(x_points))
    xmax = float(np.max(x_points))

    for idx, (label, values, color) in enumerate(diffs, start=1):
        c.cd(idx)
        ROOT.gPad.SetGridx(True)
        ROOT.gPad.SetGridy(True)
        ymax = float(np.max(np.abs(values)) * 1.15 + 1e-12)
        frame = ROOT.gPad.DrawFrame(xmin, -ymax, xmax, ymax)
        frame.SetTitle(label)
        frame.GetXaxis().SetTitle("p_{T} [GeV/c]")
        frame.GetYaxis().SetTitle("difference")

        g = ROOT.TGraph(len(x_points), x_points, values)
        g.SetName(f"g_diff_{idx}_{uid}")
        g.SetMarkerStyle(20)
        g.SetMarkerSize(0.75)
        g.SetMarkerColor(color)
        g.SetLineColor(color)
        g.SetLineWidth(2)
        g.Draw("LP SAME")
        zero = _draw_zero_line(xmin, xmax)
        keep.extend([frame, g, zero])

    c._keep = keep
    c.Modified()
    c.Update()

    if hasattr(ROOT, "JSROOT") and hasattr(ROOT.JSROOT, "Draw"):
        return ROOT.JSROOT.Draw(c)
    return c


def plot_diff_scan_heatmaps_root_inline(
    x_points,
    pointwise_diff_scan,
    *,
    title="Pointwise differences vs length scale",
    canvas_name="c_diff_scan_heatmaps",
    width=1200,
    height=900,
):
    ROOT.gROOT.SetBatch(True)
    ROOT.gStyle.SetOptStat(0)

    uid = uuid.uuid4().hex[:6]
    cname = f"{canvas_name}_{uid}"
    c = ROOT.TCanvas(cname, title, width, height)
    c.Divide(1, 3, 0.01, 0.01)

    ell_values = np.asarray(pointwise_diff_scan["ell"], dtype=float)
    maps = [
        ("GP mean - data central value", pointwise_diff_scan["mean"]),
        ("GP uncorr. - exp stat #oplus syst uncorr.", pointwise_diff_scan["uncorr"]),
        ("GP corr. - exp corr.", pointwise_diff_scan["corr"]),
    ]

    n_ell = len(ell_values)
    n_points = len(x_points)
    keep = []

    for idx, (label, values) in enumerate(maps, start=1):
        c.cd(idx)
        ROOT.gPad.SetRightMargin(0.14)
        h = ROOT.TH2D(
            f"h_diff_scan_{idx}_{uid}",
            label,
            n_points, -0.5, n_points - 0.5,
            n_ell, float(ell_values[0] - 0.05), float(ell_values[-1] + 0.05),
        )
        values = np.asarray(values, dtype=float)
        for i_ell in range(n_ell):
            for i_point in range(n_points):
                h.SetBinContent(i_point + 1, i_ell + 1, float(values[i_ell, i_point]))

        max_abs = float(np.max(np.abs(values)))
        h.SetMinimum(-max_abs)
        h.SetMaximum(max_abs)
        h.GetXaxis().SetTitle("data point")
        h.GetYaxis().SetTitle("RBF length scale ell")
        h.GetZaxis().SetTitle("difference")
        h.Draw("COLZ")
        keep.append(h)

    c._keep = keep
    c.Modified()
    c.Update()

    if hasattr(ROOT, "JSROOT") and hasattr(ROOT.JSROOT, "Draw"):
        return ROOT.JSROOT.Draw(c)
    return c


c_pointwise_diffs = plot_pointwise_diffs_root_inline(
    pT,
    gp_mean_train - y,
    sigma_uncorr_model - sigma_uncorr_exp,
    sigma_corr_model - sigma_corr_exp,
    title=f"Pointwise differences at ell = {LENGTH_SCALE}",
)
display(c_pointwise_diffs)

c_diff_scan_heatmaps = plot_diff_scan_heatmaps_root_inline(pT, pointwise_diff_scan)
display(c_diff_scan_heatmaps)


def plot_diff_scan_lines_root_inline(
    x_points,
    pointwise_diff_scan,
    *,
    title="Pointwise differences for all tested length scales",
    canvas_name="c_diff_scan_lines",
    width=1250,
    height=950,
):
    ROOT.gROOT.SetBatch(True)
    ROOT.gStyle.SetOptStat(0)

    uid = uuid.uuid4().hex[:6]
    cname = f"{canvas_name}_{uid}"
    c = ROOT.TCanvas(cname, title, width, height)
    c.Divide(1, 3, 0.01, 0.01)

    x_points = np.asarray(x_points, dtype=np.float64).ravel()
    ell_values = np.asarray(pointwise_diff_scan["ell"], dtype=float)
    maps = [
        ("GP mean - data central value", pointwise_diff_scan["mean"]),
        ("GP uncorr. - exp stat #oplus syst uncorr.", pointwise_diff_scan["uncorr"]),
        ("GP corr. - exp corr.", pointwise_diff_scan["corr"]),
    ]
    colors = [
        ROOT.kAzure + 2,
        ROOT.kOrange + 7,
        ROOT.kGreen + 2,
        ROOT.kMagenta + 1,
        ROOT.kRed + 1,
        ROOT.kCyan + 2,
        ROOT.kViolet + 6,
        ROOT.kSpring + 5,
        ROOT.kPink + 7,
        ROOT.kBlue + 3,
    ]

    xmin = float(np.min(x_points))
    xmax = float(np.max(x_points))
    keep = []

    for idx, (label, values) in enumerate(maps, start=1):
        c.cd(idx)
        ROOT.gPad.SetGridx(True)
        ROOT.gPad.SetGridy(True)
        values = np.asarray(values, dtype=float)
        ymax = float(np.max(np.abs(values)) * 1.18 + 1e-12)
        frame = ROOT.gPad.DrawFrame(xmin, -ymax, xmax, ymax)
        frame.SetTitle(label)
        frame.GetXaxis().SetTitle("p_{T} [GeV/c]")
        frame.GetYaxis().SetTitle("difference")

        zero = _draw_zero_line(xmin, xmax)
        graphs = []
        for i_ell, ell in enumerate(ell_values):
            g = ROOT.TGraph(len(x_points), x_points, values[i_ell].astype(np.float64))
            g.SetName(f"g_diff_scan_line_{idx}_{i_ell}_{uid}")
            g.SetLineColor(colors[i_ell % len(colors)])
            g.SetMarkerColor(colors[i_ell % len(colors)])
            g.SetLineWidth(2)
            g.SetMarkerStyle(20 + (i_ell % 10))
            g.SetMarkerSize(0.45)
            g.Draw("LP SAME")
            graphs.append(g)

        leg = ROOT.TLegend(0.72, 0.12, 0.90, 0.48)
        leg.SetBorderSize(0)
        leg.SetFillStyle(0)
        leg.SetNColumns(2)
        for i_ell, (ell, g) in enumerate(zip(ell_values, graphs)):
            leg.AddEntry(g, f"ell={ell:.1f}", "lp")
        leg.Draw()

        keep.extend([frame, zero, leg, *graphs])

    c._keep = keep
    c.Modified()
    c.Update()

    if hasattr(ROOT, "JSROOT") and hasattr(ROOT.JSROOT, "Draw"):
        return ROOT.JSROOT.Draw(c)
    return c


c_diff_scan_lines = plot_diff_scan_lines_root_inline(pT, pointwise_diff_scan)
display(c_diff_scan_lines)


def plot_diffs_grouped_by_length_scale_root_inline(
    x_points,
    pointwise_diff_scan,
    *,
    title="All pointwise differences grouped by length scale",
    canvas_name="c_diffs_by_ell",
    width=1500,
    height=1050,
):
    ROOT.gROOT.SetBatch(True)
    ROOT.gStyle.SetOptStat(0)

    uid = uuid.uuid4().hex[:6]
    cname = f"{canvas_name}_{uid}"
    c = ROOT.TCanvas(cname, title, width, height)
    c.Divide(2, 5, 0.01, 0.01)

    x_points = np.asarray(x_points, dtype=np.float64).ravel()
    ell_values = np.asarray(pointwise_diff_scan["ell"], dtype=float)
    diff_mean = np.asarray(pointwise_diff_scan["mean"], dtype=float)
    diff_uncorr = np.asarray(pointwise_diff_scan["uncorr"], dtype=float)
    diff_corr = np.asarray(pointwise_diff_scan["corr"], dtype=float)

    xmin = float(np.min(x_points))
    xmax = float(np.max(x_points))
    keep = []

    for i_ell, ell in enumerate(ell_values):
        c.cd(i_ell + 1)
        ROOT.gPad.SetGridx(True)
        ROOT.gPad.SetGridy(True)

        values = [diff_mean[i_ell], diff_uncorr[i_ell], diff_corr[i_ell]]
        ymax = float(max(np.max(np.abs(v)) for v in values) * 1.20 + 1e-12)
        frame = ROOT.gPad.DrawFrame(xmin, -ymax, xmax, ymax)
        frame.SetTitle(f"ell = {ell:.1f}")
        frame.GetXaxis().SetTitle("p_{T} [GeV/c]")
        frame.GetYaxis().SetTitle("difference")

        zero = _draw_zero_line(xmin, xmax)

        g_mean = ROOT.TGraph(len(x_points), x_points, diff_mean[i_ell].astype(np.float64))
        g_mean.SetName(f"g_diff_mean_ell_{i_ell}_{uid}")
        g_mean.SetLineColor(ROOT.kBlack)
        g_mean.SetMarkerColor(ROOT.kBlack)
        g_mean.SetMarkerStyle(20)
        g_mean.SetMarkerSize(0.45)
        g_mean.SetLineWidth(2)

        g_uncorr = ROOT.TGraph(len(x_points), x_points, diff_uncorr[i_ell].astype(np.float64))
        g_uncorr.SetName(f"g_diff_uncorr_ell_{i_ell}_{uid}")
        g_uncorr.SetLineColor(ROOT.kAzure + 2)
        g_uncorr.SetMarkerColor(ROOT.kAzure + 2)
        g_uncorr.SetMarkerStyle(21)
        g_uncorr.SetMarkerSize(0.45)
        g_uncorr.SetLineWidth(2)

        g_corr = ROOT.TGraph(len(x_points), x_points, diff_corr[i_ell].astype(np.float64))
        g_corr.SetName(f"g_diff_corr_ell_{i_ell}_{uid}")
        g_corr.SetLineColor(ROOT.kViolet + 6)
        g_corr.SetMarkerColor(ROOT.kViolet + 6)
        g_corr.SetMarkerStyle(22)
        g_corr.SetMarkerSize(0.45)
        g_corr.SetLineWidth(2)

        g_mean.Draw("LP SAME")
        g_uncorr.Draw("LP SAME")
        g_corr.Draw("LP SAME")

        leg = ROOT.TLegend(0.48, 0.66, 0.90, 0.88)
        leg.SetBorderSize(0)
        leg.SetFillStyle(0)
        leg.AddEntry(g_mean, "GP mean - data", "lp")
        leg.AddEntry(g_uncorr, "GP uncorr. - exp uncorr.", "lp")
        leg.AddEntry(g_corr, "GP corr. - exp corr.", "lp")
        leg.Draw()

        pad_keep = [frame, zero, g_mean, g_uncorr, g_corr, leg]

        keep.extend(pad_keep)

    c._keep = keep
    c.Modified()
    c.Update()

    if hasattr(ROOT, "JSROOT") and hasattr(ROOT.JSROOT, "Draw"):
        return ROOT.JSROOT.Draw(c)
    return c


c_diffs_by_ell = plot_diffs_grouped_by_length_scale_root_inline(pT, pointwise_diff_scan)
display(c_diffs_by_ell)


## Ratio analysis

This section repeats the point-by-point and length-scale scan analysis using ratios instead of absolute differences. A value of 1 means perfect agreement between the GP estimate and the corresponding experimental quantity.


In [10]:
def safe_ratio(numerator, denominator):
    numerator = np.asarray(numerator, dtype=float)
    denominator = np.asarray(denominator, dtype=float)
    return np.divide(
        numerator,
        denominator,
        out=np.full_like(numerator, np.nan, dtype=float),
        where=denominator != 0,
    )


ratio_mean_default = safe_ratio(gp_mean_train, y)
ratio_uncorr_default = safe_ratio(sigma_uncorr_model, sigma_uncorr_exp)
ratio_corr_default = safe_ratio(sigma_corr_model, sigma_corr_exp)

pointwise_ratio_scan = {
    "ell": pointwise_diff_scan["ell"],
    "mean": safe_ratio(pointwise_diff_scan["mean"] + y[None, :], y[None, :]),
    "uncorr": safe_ratio(pointwise_diff_scan["uncorr"] + sigma_uncorr_exp[None, :], sigma_uncorr_exp[None, :]),
    "corr": safe_ratio(pointwise_diff_scan["corr"] + sigma_corr_exp[None, :], sigma_corr_exp[None, :]),
}


def ratio_point_table_html(max_rows=None):
    n = len(pT) if max_rows is None else min(max_rows, len(pT))
    rows = []
    for i in range(n):
        rows.append(
            "<tr>"
            f"<td>{i}</td>"
            f"<td>{pT[i]:.6g}</td>"
            f"<td>{y[i]:.6g}</td>"
            f"<td>{gp_mean_train[i]:.6g}</td>"
            f"<td>{ratio_mean_default[i]:.6g}</td>"
            f"<td>{sigma_uncorr_exp[i]:.6g}</td>"
            f"<td>{sigma_uncorr_model[i]:.6g}</td>"
            f"<td>{ratio_uncorr_default[i]:.6g}</td>"
            f"<td>{sigma_corr_exp[i]:.6g}</td>"
            f"<td>{sigma_corr_model[i]:.6g}</td>"
            f"<td>{ratio_corr_default[i]:.6g}</td>"
            "</tr>"
        )

    return """
    <table>
      <thead>
        <tr>
          <th>data point</th>
          <th>pT [GeV/c]</th>
          <th>data central value</th>
          <th>GP mean</th>
          <th>ratio mean (GP / data)</th>
          <th>exp stat + uncorr syst</th>
          <th>GP uncorrelated uncertainty</th>
          <th>ratio uncorrelated (GP / exp)</th>
          <th>exp correlated uncertainty</th>
          <th>GP correlated uncertainty</th>
          <th>ratio correlated (GP / exp)</th>
        </tr>
      </thead>
      <tbody>
        {rows}
      </tbody>
    </table>
    """.format(rows="\n".join(rows))


def ratio_scan_table_html():
    rows = []
    for i, result in enumerate(length_scale_scan):
        mean_ratio = pointwise_ratio_scan["mean"][i]
        uncorr_ratio = pointwise_ratio_scan["uncorr"][i]
        corr_ratio = pointwise_ratio_scan["corr"][i]
        rows.append(
            "<tr>"
            f"<td>{result['ell']:.1f}</td>"
            f"<td>{result['sigma_f']:.6g}</td>"
            f"<td>{result['sigma_n']:.6g}</td>"
            f"<td>{np.nanmean(mean_ratio):.6g}</td>"
            f"<td>{np.nanmean(np.abs(mean_ratio - 1.0)):.6g}</td>"
            f"<td>{np.sqrt(np.nanmean((mean_ratio - 1.0)**2)):.6g}</td>"
            f"<td>{np.nanmean(uncorr_ratio):.6g}</td>"
            f"<td>{np.nanmean(np.abs(uncorr_ratio - 1.0)):.6g}</td>"
            f"<td>{np.sqrt(np.nanmean((uncorr_ratio - 1.0)**2)):.6g}</td>"
            f"<td>{np.nanmean(corr_ratio):.6g}</td>"
            f"<td>{np.nanmean(np.abs(corr_ratio - 1.0)):.6g}</td>"
            f"<td>{np.sqrt(np.nanmean((corr_ratio - 1.0)**2)):.6g}</td>"
            "</tr>"
        )

    return """
    <table>
      <thead>
        <tr>
          <th>ell</th>
          <th>sigma_f log(y)</th>
          <th>sigma_n log(y)</th>
          <th>mean ratio GP mean</th>
          <th>mean abs ratio-1 GP mean</th>
          <th>RMS ratio-1 GP mean</th>
          <th>mean ratio uncorr</th>
          <th>mean abs ratio-1 uncorr</th>
          <th>RMS ratio-1 uncorr</th>
          <th>mean ratio corr</th>
          <th>mean abs ratio-1 corr</th>
          <th>RMS ratio-1 corr</th>
        </tr>
      </thead>
      <tbody>
        {rows}
      </tbody>
    </table>
    """.format(rows="\n".join(rows))


display(HTML(ratio_point_table_html()))
display(HTML(ratio_scan_table_html()))


import uuid
import ROOT


def _draw_one_line(xmin, xmax):
    line = ROOT.TLine(xmin, 1.0, xmax, 1.0)
    line.SetLineStyle(2)
    line.SetLineColor(ROOT.kGray + 2)
    line.SetLineWidth(1)
    line.Draw("SAME")
    return line


def plot_pointwise_ratios_root_inline(
    x_points,
    ratio_mean,
    ratio_uncorr,
    ratio_corr,
    *,
    title="Pointwise ratios for default length scale",
    canvas_name="c_pointwise_ratios",
    width=1050,
    height=900,
):
    ROOT.gROOT.SetBatch(True)
    ROOT.gStyle.SetOptStat(0)

    uid = uuid.uuid4().hex[:6]
    c = ROOT.TCanvas(f"{canvas_name}_{uid}", title, width, height)
    c.Divide(1, 3, 0.01, 0.01)

    x_points = np.asarray(x_points, dtype=np.float64).ravel()
    ratios = [
        ("GP mean / data central value", np.asarray(ratio_mean, dtype=np.float64), ROOT.kBlack),
        ("GP uncorr. / exp stat #oplus syst uncorr.", np.asarray(ratio_uncorr, dtype=np.float64), ROOT.kAzure + 2),
        ("GP corr. / exp corr.", np.asarray(ratio_corr, dtype=np.float64), ROOT.kViolet + 6),
    ]

    keep = []
    xmin = float(np.min(x_points))
    xmax = float(np.max(x_points))

    for idx, (label, values, color) in enumerate(ratios, start=1):
        c.cd(idx)
        ROOT.gPad.SetGridx(True)
        ROOT.gPad.SetGridy(True)
        finite = values[np.isfinite(values)]
        ymin = float(min(np.min(finite), 1.0) * 0.92)
        ymax = float(max(np.max(finite), 1.0) * 1.08)
        frame = ROOT.gPad.DrawFrame(xmin, ymin, xmax, ymax)
        frame.SetTitle(label)
        frame.GetXaxis().SetTitle("p_{T} [GeV/c]")
        frame.GetYaxis().SetTitle("ratio")

        g = ROOT.TGraph(len(x_points), x_points, values)
        g.SetName(f"g_ratio_{idx}_{uid}")
        g.SetMarkerStyle(20)
        g.SetMarkerSize(0.75)
        g.SetMarkerColor(color)
        g.SetLineColor(color)
        g.SetLineWidth(2)
        g.Draw("LP SAME")
        one = _draw_one_line(xmin, xmax)
        keep.extend([frame, g, one])

    c._keep = keep
    c.Modified()
    c.Update()
    if hasattr(ROOT, "JSROOT") and hasattr(ROOT.JSROOT, "Draw"):
        return ROOT.JSROOT.Draw(c)
    return c


def plot_ratio_scan_heatmaps_root_inline(
    x_points,
    pointwise_ratio_scan,
    *,
    title="Pointwise ratios vs length scale",
    canvas_name="c_ratio_scan_heatmaps",
    width=1200,
    height=900,
):
    ROOT.gROOT.SetBatch(True)
    ROOT.gStyle.SetOptStat(0)

    uid = uuid.uuid4().hex[:6]
    c = ROOT.TCanvas(f"{canvas_name}_{uid}", title, width, height)
    c.Divide(1, 3, 0.01, 0.01)

    ell_values = np.asarray(pointwise_ratio_scan["ell"], dtype=float)
    maps = [
        ("GP mean / data central value", pointwise_ratio_scan["mean"]),
        ("GP uncorr. / exp stat #oplus syst uncorr.", pointwise_ratio_scan["uncorr"]),
        ("GP corr. / exp corr.", pointwise_ratio_scan["corr"]),
    ]

    n_ell = len(ell_values)
    n_points = len(x_points)
    keep = []

    for idx, (label, values) in enumerate(maps, start=1):
        c.cd(idx)
        ROOT.gPad.SetRightMargin(0.14)
        h = ROOT.TH2D(
            f"h_ratio_scan_{idx}_{uid}",
            label,
            n_points, -0.5, n_points - 0.5,
            n_ell, float(ell_values[0] - 0.05), float(ell_values[-1] + 0.05),
        )
        values = np.asarray(values, dtype=float)
        for i_ell in range(n_ell):
            for i_point in range(n_points):
                h.SetBinContent(i_point + 1, i_ell + 1, float(values[i_ell, i_point]))

        max_dev = float(np.nanmax(np.abs(values - 1.0)))
        h.SetMinimum(1.0 - max_dev)
        h.SetMaximum(1.0 + max_dev)
        h.GetXaxis().SetTitle("data point")
        h.GetYaxis().SetTitle("RBF length scale ell")
        h.GetZaxis().SetTitle("ratio")
        h.Draw("COLZ")
        keep.append(h)

    c._keep = keep
    c.Modified()
    c.Update()
    if hasattr(ROOT, "JSROOT") and hasattr(ROOT.JSROOT, "Draw"):
        return ROOT.JSROOT.Draw(c)
    return c


def plot_ratio_scan_lines_root_inline(
    x_points,
    pointwise_ratio_scan,
    *,
    title="Pointwise ratios for all tested length scales",
    canvas_name="c_ratio_scan_lines",
    width=1250,
    height=950,
):
    ROOT.gROOT.SetBatch(True)
    ROOT.gStyle.SetOptStat(0)

    uid = uuid.uuid4().hex[:6]
    c = ROOT.TCanvas(f"{canvas_name}_{uid}", title, width, height)
    c.Divide(1, 3, 0.01, 0.01)

    x_points = np.asarray(x_points, dtype=np.float64).ravel()
    ell_values = np.asarray(pointwise_ratio_scan["ell"], dtype=float)
    maps = [
        ("GP mean / data central value", pointwise_ratio_scan["mean"]),
        ("GP uncorr. / exp stat #oplus syst uncorr.", pointwise_ratio_scan["uncorr"]),
        ("GP corr. / exp corr.", pointwise_ratio_scan["corr"]),
    ]
    colors = [ROOT.kAzure + 2, ROOT.kOrange + 7, ROOT.kGreen + 2, ROOT.kMagenta + 1, ROOT.kRed + 1,
              ROOT.kCyan + 2, ROOT.kViolet + 6, ROOT.kSpring + 5, ROOT.kPink + 7, ROOT.kBlue + 3]

    xmin = float(np.min(x_points))
    xmax = float(np.max(x_points))
    keep = []

    for idx, (label, values) in enumerate(maps, start=1):
        c.cd(idx)
        ROOT.gPad.SetGridx(True)
        ROOT.gPad.SetGridy(True)
        values = np.asarray(values, dtype=float)
        max_dev = float(np.nanmax(np.abs(values - 1.0)))
        frame = ROOT.gPad.DrawFrame(xmin, 1.0 - 1.18 * max_dev, xmax, 1.0 + 1.18 * max_dev)
        frame.SetTitle(label)
        frame.GetXaxis().SetTitle("p_{T} [GeV/c]")
        frame.GetYaxis().SetTitle("ratio")
        one = _draw_one_line(xmin, xmax)

        graphs = []
        for i_ell, ell in enumerate(ell_values):
            g = ROOT.TGraph(len(x_points), x_points, values[i_ell].astype(np.float64))
            g.SetName(f"g_ratio_scan_line_{idx}_{i_ell}_{uid}")
            g.SetLineColor(colors[i_ell % len(colors)])
            g.SetMarkerColor(colors[i_ell % len(colors)])
            g.SetLineWidth(2)
            g.SetMarkerStyle(20 + (i_ell % 10))
            g.SetMarkerSize(0.45)
            g.Draw("LP SAME")
            graphs.append(g)

        leg = ROOT.TLegend(0.72, 0.12, 0.90, 0.48)
        leg.SetBorderSize(0)
        leg.SetFillStyle(0)
        leg.SetNColumns(2)
        for ell, g in zip(ell_values, graphs):
            leg.AddEntry(g, f"ell={ell:.1f}", "lp")
        leg.Draw()
        keep.extend([frame, one, leg, *graphs])

    c._keep = keep
    c.Modified()
    c.Update()
    if hasattr(ROOT, "JSROOT") and hasattr(ROOT.JSROOT, "Draw"):
        return ROOT.JSROOT.Draw(c)
    return c


def plot_ratios_grouped_by_length_scale_root_inline(
    x_points,
    pointwise_ratio_scan,
    *,
    title="All pointwise ratios grouped by length scale",
    canvas_name="c_ratios_by_ell",
    width=1500,
    height=1050,
):
    ROOT.gROOT.SetBatch(True)
    ROOT.gStyle.SetOptStat(0)

    uid = uuid.uuid4().hex[:6]
    c = ROOT.TCanvas(f"{canvas_name}_{uid}", title, width, height)
    c.Divide(2, 5, 0.01, 0.01)

    x_points = np.asarray(x_points, dtype=np.float64).ravel()
    ell_values = np.asarray(pointwise_ratio_scan["ell"], dtype=float)
    ratio_mean = np.asarray(pointwise_ratio_scan["mean"], dtype=float)
    ratio_uncorr = np.asarray(pointwise_ratio_scan["uncorr"], dtype=float)
    ratio_corr = np.asarray(pointwise_ratio_scan["corr"], dtype=float)

    xmin = float(np.min(x_points))
    xmax = float(np.max(x_points))
    keep = []

    for i_ell, ell in enumerate(ell_values):
        c.cd(i_ell + 1)
        ROOT.gPad.SetGridx(True)
        ROOT.gPad.SetGridy(True)
        values = [ratio_mean[i_ell], ratio_uncorr[i_ell], ratio_corr[i_ell]]
        max_dev = float(max(np.nanmax(np.abs(v - 1.0)) for v in values))
        frame = ROOT.gPad.DrawFrame(xmin, 1.0 - 1.20 * max_dev, xmax, 1.0 + 1.20 * max_dev)
        frame.SetTitle(f"ell = {ell:.1f}")
        frame.GetXaxis().SetTitle("p_{T} [GeV/c]")
        frame.GetYaxis().SetTitle("ratio")
        one = _draw_one_line(xmin, xmax)

        g_mean = ROOT.TGraph(len(x_points), x_points, ratio_mean[i_ell].astype(np.float64))
        g_mean.SetName(f"g_ratio_mean_ell_{i_ell}_{uid}")
        g_mean.SetLineColor(ROOT.kBlack)
        g_mean.SetMarkerColor(ROOT.kBlack)
        g_mean.SetMarkerStyle(20)
        g_mean.SetMarkerSize(0.45)
        g_mean.SetLineWidth(2)

        g_uncorr = ROOT.TGraph(len(x_points), x_points, ratio_uncorr[i_ell].astype(np.float64))
        g_uncorr.SetName(f"g_ratio_uncorr_ell_{i_ell}_{uid}")
        g_uncorr.SetLineColor(ROOT.kAzure + 2)
        g_uncorr.SetMarkerColor(ROOT.kAzure + 2)
        g_uncorr.SetMarkerStyle(21)
        g_uncorr.SetMarkerSize(0.45)
        g_uncorr.SetLineWidth(2)

        g_corr = ROOT.TGraph(len(x_points), x_points, ratio_corr[i_ell].astype(np.float64))
        g_corr.SetName(f"g_ratio_corr_ell_{i_ell}_{uid}")
        g_corr.SetLineColor(ROOT.kViolet + 6)
        g_corr.SetMarkerColor(ROOT.kViolet + 6)
        g_corr.SetMarkerStyle(22)
        g_corr.SetMarkerSize(0.45)
        g_corr.SetLineWidth(2)

        g_mean.Draw("LP SAME")
        g_uncorr.Draw("LP SAME")
        g_corr.Draw("LP SAME")

        leg = ROOT.TLegend(0.48, 0.66, 0.90, 0.88)
        leg.SetBorderSize(0)
        leg.SetFillStyle(0)
        leg.AddEntry(g_mean, "GP mean / data", "lp")
        leg.AddEntry(g_uncorr, "GP uncorr. / exp uncorr.", "lp")
        leg.AddEntry(g_corr, "GP corr. / exp corr.", "lp")
        leg.Draw()

        pad_keep = [frame, one, g_mean, g_uncorr, g_corr, leg]

        keep.extend(pad_keep)

    c._keep = keep
    c.Modified()
    c.Update()
    if hasattr(ROOT, "JSROOT") and hasattr(ROOT.JSROOT, "Draw"):
        return ROOT.JSROOT.Draw(c)
    return c


c_pointwise_ratios = plot_pointwise_ratios_root_inline(
    pT, ratio_mean_default, ratio_uncorr_default, ratio_corr_default,
    title=f"Pointwise ratios at ell = {LENGTH_SCALE}",
)
display(c_pointwise_ratios)

c_ratio_scan_heatmaps = plot_ratio_scan_heatmaps_root_inline(pT, pointwise_ratio_scan)
display(c_ratio_scan_heatmaps)

c_ratio_scan_lines = plot_ratio_scan_lines_root_inline(pT, pointwise_ratio_scan)
display(c_ratio_scan_lines)

c_ratios_by_ell = plot_ratios_grouped_by_length_scale_root_inline(pT, pointwise_ratio_scan)
display(c_ratios_by_ell)


data point,pT [GeV/c],data central value,GP mean,ratio mean (GP / data),exp stat + uncorr syst,GP uncorrelated uncertainty,ratio uncorrelated (GP / exp),exp correlated uncertainty,GP correlated uncertainty,ratio correlated (GP / exp)
0,0.11,2049.8,2042.98,0.996673,41.3612,23.1022,0.558547,140.449,21.2099,0.151014
1,0.13,2187.3,2200.79,1.00617,43.7268,24.8867,0.56914,103.645,14.5905,0.140774
2,0.15,2291.6,2299.66,1.00352,45.7113,26.0048,0.568892,103.267,15.3286,0.148437
3,0.17,2357.6,2349.66,0.996634,46.968,26.5702,0.565708,102.111,14.8625,0.145553
4,0.19,2384.7,2365.2,0.991824,47.4732,26.7459,0.563389,102.53,15.1096,0.147367
5,0.225,2356.9,2350.2,0.997159,46.7014,26.5763,0.569069,102.399,17.5442,0.171332
6,0.275,2250.6,2307.49,1.02528,44.5992,26.0933,0.585062,100.729,17.8949,0.177654
7,0.325,2306,2249.92,0.97568,28.7158,25.4423,0.886005,47.3135,17.3184,0.366034
8,0.375,2131.4,2135.8,1.00207,23.0206,24.1518,1.04914,42.1983,16.2091,0.384118
9,0.425,1937.2,1955.69,1.00955,20.579,22.1151,1.07465,39.8049,14.7222,0.369859


ell,sigma_f log(y),sigma_n log(y),mean ratio GP mean,mean abs ratio-1 GP mean,RMS ratio-1 GP mean,mean ratio uncorr,mean abs ratio-1 uncorr,RMS ratio-1 uncorr,mean ratio corr,mean abs ratio-1 corr,RMS ratio-1 corr
0.1,4.26646,0.00576154,1,0.00059144,0.0014053,0.413056,0.586944,0.61103,0.210437,0.789563,0.791497
0.2,4.56018,0.0113081,1.00003,0.00237531,0.00537426,0.810658,0.333143,0.383358,0.371162,0.628838,0.640296
0.3,4.73727,0.0143729,1.00005,0.00423859,0.00820576,1.03028,0.341113,0.424526,0.439271,0.560729,0.586691
0.4,4.80583,0.0150705,1.00006,0.00482168,0.00953506,1.08031,0.349356,0.451205,0.434207,0.565793,0.596173
0.5,4.6829,0.0153487,1.00007,0.00537183,0.0103715,1.10026,0.352784,0.463183,0.419424,0.580576,0.611767
0.6,4.6582,0.0149154,1.00007,0.00559796,0.0104894,1.06926,0.347688,0.445007,0.391014,0.608986,0.637514
0.7,4.83183,0.014923,1.00007,0.00608715,0.0107821,1.0699,0.348185,0.445627,0.37797,0.62203,0.650266
0.8,5.14763,0.0158997,1.00008,0.00695407,0.011712,1.14009,0.363111,0.489917,0.390533,0.609467,0.64218
0.9,5.49316,0.0176065,1.0001,0.00835045,0.0131837,1.26273,0.395599,0.583071,0.420021,0.579979,0.621524
1.0,5.59016,0.0202255,1.00013,0.0101049,0.0154204,1.45081,0.489908,0.749391,0.468147,0.531853,0.589484


## GP fit with optimized correlation length

This section repeats the GP fit with `sigma_f`, `ell`, and `sigma_n` optimized together. The experimental uncertainty columns are still used only for comparison, not as inputs to the GP fit.


In [17]:
def negative_log_marginal_likelihood_free_ell(theta):
    log_sigma_f, log_ell, log_sigma_n = theta
    sigma_f_free = np.exp(log_sigma_f)
    ell_free = np.exp(log_ell)
    sigma_n_free = np.exp(log_sigma_n)
    R_free = rbf_correlation(pT, pT, length_scale=ell_free)
    K_free = sigma_f_free**2 * R_free + sigma_n_free**2 * np.eye(len(pT))
    L_free = stable_cholesky(K_free)
    alpha_free = np.linalg.solve(L_free.T, np.linalg.solve(L_free, z))
    log_det_free = 2.0 * np.sum(np.log(np.diag(L_free)))
    return 0.5 * z @ alpha_free + 0.5 * log_det_free + 0.5 * len(pT) * np.log(2.0 * np.pi)


def fit_gp_with_free_length_scale():
    initial = np.log([sigma_f, LENGTH_SCALE, sigma_n])
    bounds = [
        (np.log(1e-4 * np.std(z)), np.log(20.0 * np.std(z))),
        (np.log(0.02), np.log(10.0)),
        (np.log(1e-6 * np.std(z)), np.log(5.0 * np.std(z))),
    ]

    if minimize is not None:
        starts = [
            initial,
            np.log([np.std(z), 0.1, 0.01 * np.std(z)]),
            np.log([np.std(z), 0.5, 0.05 * np.std(z)]),
            np.log([2.0 * np.std(z), 1.0, 0.10 * np.std(z)]),
        ]
        results = []
        for start in starts:
            result = minimize(
                negative_log_marginal_likelihood_free_ell,
                start,
                method="L-BFGS-B",
                bounds=bounds,
                options={"maxiter": 10000, "ftol": 1e-12, "gtol": 1e-8},
            )
            if result.success and np.isfinite(result.fun):
                results.append(result)

        if results:
            result = min(results, key=lambda item: item.fun)
            sigma_f_best, ell_best, sigma_n_best = np.exp(result.x)
            R_best = rbf_correlation(pT, pT, length_scale=ell_best)
            K_corr_best = sigma_f_best**2 * R_best
            K_delta_best = sigma_n_best**2 * np.eye(len(pT))
            return {
                "length_scale": ell_best,
                "R": R_best,
                "sigma_f": sigma_f_best,
                "sigma_n": sigma_n_best,
                "nll": result.fun,
                "fit_method": "scipy L-BFGS-B, free ell",
                "K_corr_log": K_corr_best,
                "K_delta_log": K_delta_best,
                "K_total_log": K_corr_best + K_delta_best,
            }

    ell_grid = np.round(np.arange(0.1, 1.0 + 0.05, 0.1), 1)
    best = None
    for ell in ell_grid:
        fit = fit_gp_for_length_scale(float(ell))
        if best is None or fit["nll"] < best["nll"]:
            best = dict(fit)
    best["fit_method"] = "profile scan fallback over ell = 0.1 ... 1.0"
    return best


fit_free_ell = fit_gp_with_free_length_scale()
gp_mean_free, sigma_corr_free, sigma_uncorr_free = gp_uncertainties_at_training_points(fit_free_ell)

ell_free = fit_free_ell["length_scale"]
sigma_f_free = fit_free_ell["sigma_f"]
sigma_n_free = fit_free_ell["sigma_n"]
nll_free = fit_free_ell["nll"]

relative_exp_uncorr = safe_ratio(sigma_uncorr_exp, y)
relative_exp_corr = safe_ratio(sigma_corr_exp, y)
relative_gp_uncorr_free = safe_ratio(sigma_uncorr_free, gp_mean_free)
relative_gp_corr_free = safe_ratio(sigma_corr_free, gp_mean_free)

print(f"Free-ell GP fit method: {fit_free_ell['fit_method']}")
print(f"optimal ell: {ell_free:.6g}")
print(f"sigma_f on log(y): {sigma_f_free:.6g}")
print(f"sigma_n on log(y): {sigma_n_free:.6g}")
print(f"negative log marginal likelihood: {nll_free:.6g}")


def free_ell_hyperparameter_table_html():
    return f"""
    <table>
      <thead>
        <tr>
          <th>quantity</th>
          <th>GP free-ell fit</th>
          <th>experimental reference</th>
          <th>comment</th>
        </tr>
      </thead>
      <tbody>
        <tr>
          <td>RBF length scale ell</td>
          <td>{ell_free:.6g}</td>
          <td>-</td>
          <td>Optimized from central values by marginal likelihood.</td>
        </tr>
        <tr>
          <td>kernel amplitude sigma_f on log(y)</td>
          <td>{sigma_f_free:.6g}</td>
          <td>mean exp corr / y = {np.nanmean(relative_exp_corr):.6g}</td>
          <td>sigma_f is a prior amplitude in log-space; compare cautiously with pointwise relative correlated uncertainty.</td>
        </tr>
        <tr>
          <td>kernel variance sigma_f^2 on log(y)</td>
          <td>{sigma_f_free**2:.6g}</td>
          <td>mean (exp corr / y)^2 = {np.nanmean(relative_exp_corr**2):.6g}</td>
          <td>Variance-level comparison for the RBF amplitude.</td>
        </tr>
        <tr>
          <td>white-noise amplitude sigma_n on log(y)</td>
          <td>{sigma_n_free:.6g}</td>
          <td>mean exp uncorr / y = {np.nanmean(relative_exp_uncorr):.6g}</td>
          <td>For log(y), sigma_n is approximately a fractional uncorrelated uncertainty.</td>
        </tr>
        <tr>
          <td>white-noise variance sigma_n^2 on log(y)</td>
          <td>{sigma_n_free**2:.6g}</td>
          <td>mean (exp uncorr / y)^2 = {np.nanmean(relative_exp_uncorr**2):.6g}</td>
          <td>Variance-level comparison for the diagonal noise term.</td>
        </tr>
        <tr>
          <td>NLL</td>
          <td>{nll_free:.6g}</td>
          <td>fixed ell=0.2 NLL = {nll:.6g}</td>
          <td>Lower NLL is preferred by the GP marginal likelihood.</td>
        </tr>
      </tbody>
    </table>
    """


def free_ell_point_table_html(max_rows=None):
    n = len(pT) if max_rows is None else min(max_rows, len(pT))
    rows = []
    for i in range(n):
        ratio_mean = safe_ratio(gp_mean_free[i], y[i])
        ratio_uncorr = safe_ratio(sigma_uncorr_free[i], sigma_uncorr_exp[i])
        ratio_corr = safe_ratio(sigma_corr_free[i], sigma_corr_exp[i])
        rows.append(
            "<tr>"
            f"<td>{i}</td>"
            f"<td>{pT[i]:.6g}</td>"
            f"<td>{y[i]:.6g}</td>"
            f"<td>{gp_mean_free[i]:.6g}</td>"
            f"<td>{ratio_mean:.6g}</td>"
            f"<td>{sigma_uncorr_exp[i]:.6g}</td>"
            f"<td>{sigma_uncorr_free[i]:.6g}</td>"
            f"<td>{ratio_uncorr:.6g}</td>"
            f"<td>{sigma_corr_exp[i]:.6g}</td>"
            f"<td>{sigma_corr_free[i]:.6g}</td>"
            f"<td>{ratio_corr:.6g}</td>"
            "</tr>"
        )
    return """
    <table>
      <thead>
        <tr>
          <th>data point</th>
          <th>pT [GeV/c]</th>
          <th>data central value</th>
          <th>GP mean, free ell</th>
          <th>ratio mean (GP / data)</th>
          <th>exp stat + uncorr syst</th>
          <th>GP uncorrelated uncertainty</th>
          <th>ratio uncorrelated (GP / exp)</th>
          <th>exp correlated uncertainty</th>
          <th>GP correlated uncertainty</th>
          <th>ratio correlated (GP / exp)</th>
        </tr>
      </thead>
      <tbody>
        {rows}
      </tbody>
    </table>
    """.format(rows="\n".join(rows))


display(HTML(free_ell_hyperparameter_table_html()))
display(HTML(free_ell_point_table_html()))


x_grid_free = np.linspace(pT.min(), pT.max(), 600)
R_star_free = rbf_correlation(x_grid_free, pT, length_scale=ell_free)
K_star_free = sigma_f_free**2 * R_star_free
L_free = stable_cholesky(fit_free_ell["K_total_log"])
alpha_free = np.linalg.solve(L_free.T, np.linalg.solve(L_free, z))
log_mean_grid_free = np.mean(log_y) + K_star_free @ alpha_free
v_free = np.linalg.solve(L_free, K_star_free.T)
log_corr_var_grid_free = np.maximum(sigma_f_free**2 - np.sum(v_free**2, axis=0), 0.0)
gp_mean_grid_free = np.exp(log_mean_grid_free)
corr_band_grid_free = gp_mean_grid_free * np.sqrt(log_corr_var_grid_free)
exp_central_grid_free = np.exp(np.interp(x_grid_free, pT, log_y))
exp_corr_band_grid_free = np.interp(x_grid_free, pT, sigma_corr_exp)

c_gp_unc_free_ell = plot_gp_uncertainty_root_inline(
    pT, y,
    sigma_uncorr_exp, sigma_uncorr_free,
    x_grid_free, gp_mean_grid_free, corr_band_grid_free,
    exp_central_grid_free, exp_corr_band_grid_free,
    title=f"Experimental and GP uncertainty components: free ell = {ell_free:.4g}",
    canvas_name="c_gp_unc_free_ell",
)
display(c_gp_unc_free_ell)


Free-ell GP fit method: scipy L-BFGS-B, free ell
optimal ell: 3.92803
sigma_f on log(y): 6.52491
sigma_n on log(y): 0.0743027
negative log marginal likelihood: -30.8651


quantity,GP free-ell fit,experimental reference,comment
RBF length scale ell,3.92803,-,Optimized from central values by marginal likelihood.
kernel amplitude sigma_f on log(y),6.52491,mean exp corr / y = 0.0288068,sigma_f is a prior amplitude in log-space; compare cautiously with pointwise relative correlated uncertainty.
kernel variance sigma_f^2 on log(y),42.5745,mean (exp corr / y)^2 = 0.000917637,Variance-level comparison for the RBF amplitude.
white-noise amplitude sigma_n on log(y),0.0743027,mean exp uncorr / y = 0.015936,"For log(y), sigma_n is approximately a fractional uncorrelated uncertainty."
white-noise variance sigma_n^2 on log(y),0.00552089,mean (exp uncorr / y)^2 = 0.000280576,Variance-level comparison for the diagonal noise term.
NLL,-30.8651,fixed ell=0.2 NLL = 63.02,Lower NLL is preferred by the GP marginal likelihood.


data point,pT [GeV/c],data central value,"GP mean, free ell",ratio mean (GP / data),exp stat + uncorr syst,GP uncorrelated uncertainty,ratio uncorrelated (GP / exp),exp correlated uncertainty,GP correlated uncertainty,ratio correlated (GP / exp)
0,0.11,2049.8,2698.15,1.3163,41.3612,200.48,4.84705,140.449,75.1535,0.535093
1,0.13,2187.3,2625.49,1.20033,43.7268,195.081,4.46135,103.645,69.8636,0.674067
2,0.15,2291.6,2553.63,1.11434,45.7113,189.741,4.15086,103.267,64.957,0.629021
3,0.17,2357.6,2482.62,1.05303,46.968,184.466,3.92747,102.111,60.4172,0.591683
4,0.19,2384.7,2412.53,1.01167,47.4732,179.257,3.77597,102.53,56.2278,0.548404
5,0.225,2356.9,2292.2,0.972548,46.7014,170.316,3.64692,102.399,49.69,0.48526
6,0.275,2250.6,2125.82,0.944558,44.5992,157.954,3.54164,100.729,41.945,0.416415
7,0.325,2306,1966.44,0.852749,28.7158,146.112,5.0882,47.3135,35.8325,0.757342
8,0.375,2131.4,1814.47,0.851303,23.0206,134.82,5.85648,42.1983,31.0758,0.736423
9,0.425,1937.2,1670.21,0.862176,20.579,124.101,6.03046,39.8049,27.3987,0.688324


In [19]:
import uuid
import ROOT


ratio_mean_free_ell = safe_ratio(gp_mean_free, y)
ratio_uncorr_free_ell = safe_ratio(sigma_uncorr_free, sigma_uncorr_exp)
ratio_corr_free_ell = safe_ratio(sigma_corr_free, sigma_corr_exp)


def plot_free_ell_ratio_panels_root_inline(
    x_points,
    ratio_mean,
    ratio_uncorr,
    ratio_corr,
    *,
    title="Pointwise ratios for GP fit with optimized ell",
    canvas_name="c_free_ell_ratios",
    width=1050,
    height=900,
):
    ROOT.gROOT.SetBatch(True)
    ROOT.gStyle.SetOptStat(0)

    uid = uuid.uuid4().hex[:6]
    c = ROOT.TCanvas(f"{canvas_name}_{uid}", title, width, height)
    c.Divide(1, 3, 0.01, 0.01)

    x_points = np.asarray(x_points, dtype=np.float64).ravel()
    ratios = [
        ("GP mean / data central value", np.asarray(ratio_mean, dtype=np.float64), ROOT.kBlack),
        ("GP uncorr. / exp stat #oplus syst uncorr.", np.asarray(ratio_uncorr, dtype=np.float64), ROOT.kAzure + 2),
        ("GP corr. / exp corr.", np.asarray(ratio_corr, dtype=np.float64), ROOT.kAzure + 2),
    ]

    xmin = float(np.min(x_points))
    xmax = float(np.max(x_points))
    keep = []

    for idx, (label, values, color) in enumerate(ratios, start=1):
        c.cd(idx)
        ROOT.gPad.SetGridx(True)
        ROOT.gPad.SetGridy(True)

        finite = values[np.isfinite(values)]
        max_dev = float(np.max(np.abs(finite - 1.0)))
        ymin = 1.0 - 1.20 * max_dev
        ymax = 1.0 + 1.20 * max_dev
        if ymin == ymax:
            ymin, ymax = 0.9, 1.1

        frame = ROOT.gPad.DrawFrame(xmin, ymin, xmax, ymax)
        frame.SetTitle(label)
        frame.GetXaxis().SetTitle("p_{T} [GeV/c]")
        frame.GetYaxis().SetTitle("ratio")

        g = ROOT.TGraph(len(x_points), x_points, values)
        g.SetName(f"g_free_ell_ratio_{idx}_{uid}")
        g.SetMarkerStyle(20)
        g.SetMarkerSize(0.75)
        g.SetMarkerColor(color)
        g.SetLineColor(color)
        g.SetLineWidth(2)
        g.Draw("LP SAME")

        one = ROOT.TLine(xmin, 1.0, xmax, 1.0)
        one.SetLineStyle(2)
        one.SetLineColor(ROOT.kGray + 2)
        one.SetLineWidth(1)
        one.Draw("SAME")

        keep.extend([frame, g, one])

    c._keep = keep
    c.Modified()
    c.Update()

    if hasattr(ROOT, "JSROOT") and hasattr(ROOT.JSROOT, "Draw"):
        return ROOT.JSROOT.Draw(c)
    return c


def plot_free_ell_ratio_curves_root_inline(
    x_points,
    ratio_mean,
    ratio_uncorr,
    ratio_corr,
    *,
    title="Pointwise ratios for GP fit with optimized ell",
    canvas_name="c_free_ell_ratios",
    width=1050,
    height=620,
):
    ROOT.gROOT.SetBatch(True)
    ROOT.gStyle.SetOptStat(0)

    uid = uuid.uuid4().hex[:6]
    c = ROOT.TCanvas(f"{canvas_name}_{uid}", title, width, height)

    x_points = np.asarray(x_points, dtype=np.float64).ravel()
    ratios = [
        ("GP mean / data central value", np.asarray(ratio_mean, dtype=np.float64), ROOT.kBlack, 20),
        ("GP uncorr. / exp stat #oplus syst uncorr.", np.asarray(ratio_uncorr, dtype=np.float64), ROOT.kAzure + 2, 21),
        ("GP corr. / exp corr.", np.asarray(ratio_corr, dtype=np.float64), ROOT.kAzure - 4, 22),
    ]

    xmin = float(np.min(x_points))
    xmax = float(np.max(x_points))
    finite_values = np.concatenate([values[np.isfinite(values)] for _, values, _, _ in ratios])
    max_dev = float(np.max(np.abs(finite_values - 1.0)))
    ymin = 1.0 - 1.20 * max_dev
    ymax = 1.0 + 1.20 * max_dev
    if ymin == ymax:
        ymin, ymax = 0.9, 1.1

    ROOT.gPad.SetGridx(True)
    ROOT.gPad.SetGridy(True)
    frame = c.DrawFrame(xmin, ymin, xmax, ymax)
    frame.SetTitle(title)
    frame.GetXaxis().SetTitle("p_{T} [GeV/c]")
    frame.GetYaxis().SetTitle("ratio")

    one = ROOT.TLine(xmin, 1.0, xmax, 1.0)
    one.SetLineStyle(2)
    one.SetLineColor(ROOT.kGray + 2)
    one.SetLineWidth(1)
    one.Draw("SAME")

    leg = ROOT.TLegend(0.50, 0.68, 0.88, 0.88)
    leg.SetBorderSize(0)
    leg.SetFillStyle(0)
    keep = [frame, one, leg]

    for idx, (label, values, color, marker) in enumerate(ratios, start=1):
        g = ROOT.TGraph(len(x_points), x_points, values)
        g.SetName(f"g_free_ell_ratio_combined_{idx}_{uid}")
        g.SetMarkerStyle(marker)
        g.SetMarkerSize(0.75)
        g.SetMarkerColor(color)
        g.SetLineColor(color)
        g.SetLineWidth(2)
        g.Draw("LP SAME")
        leg.AddEntry(g, label, "lp")
        keep.append(g)

    leg.Draw()
    c._keep = keep
    c.Modified()
    c.Update()

    if hasattr(ROOT, "JSROOT") and hasattr(ROOT.JSROOT, "Draw"):
        return ROOT.JSROOT.Draw(c)
    return c


c_free_ell_ratio_panels = plot_free_ell_ratio_panels_root_inline(
    pT,
    ratio_mean_free_ell,
    ratio_uncorr_free_ell,
    ratio_corr_free_ell,
    title=f"Pointwise ratios for GP fit with optimized ell = {ell_free:.4g}",
)
display(c_free_ell_ratio_panels)

c_free_ell_ratio_curves = plot_free_ell_ratio_curves_root_inline(
    pT,
    ratio_mean_free_ell,
    ratio_uncorr_free_ell,
    ratio_corr_free_ell,
    title=f"Pointwise ratios for GP fit with optimized ell = {ell_free:.4g}",
)
display(c_free_ell_ratio_curves)


In [21]:
import uuid
import ROOT


def _make_filled_band_graph(x, y_low, y_high, name, color, alpha):
    x = np.asarray(x, dtype=np.float64).ravel()
    y_low = np.asarray(y_low, dtype=np.float64).ravel()
    y_high = np.asarray(y_high, dtype=np.float64).ravel()
    x_poly = np.concatenate([x, x[::-1]])
    y_poly = np.concatenate([y_high, y_low[::-1]])
    g = ROOT.TGraph(len(x_poly), x_poly.astype(np.float64), y_poly.astype(np.float64))
    g.SetName(name)
    g.SetFillStyle(1001)
    g.SetFillColorAlpha(color, alpha)
    g.SetLineColorAlpha(color, 0.0)
    return g


def plot_uncorr_ratio_and_exp_composition_root_inline(
    x_points,
    pointwise_ratio_scan,
    stat_exp,
    syst_uncorr_exp,
    *,
    title="Uncorrelated uncertainty: GP ratios and experimental composition",
    canvas_name="c_uncorr_ratio_composition",
    width=1250,
    height=850,
):
    ROOT.gROOT.SetBatch(True)
    ROOT.gStyle.SetOptStat(0)

    uid = uuid.uuid4().hex[:6]
    c = ROOT.TCanvas(f"{canvas_name}_{uid}", title, width, height)
    c.Divide(1, 2, 0.01, 0.01)

    x_points = np.asarray(x_points, dtype=np.float64).ravel()
    ell_values = np.asarray(pointwise_ratio_scan["ell"], dtype=float)
    uncorr_ratios = np.asarray(pointwise_ratio_scan["uncorr"], dtype=float)
    stat_exp = np.asarray(stat_exp, dtype=float).ravel()
    syst_uncorr_exp = np.asarray(syst_uncorr_exp, dtype=float).ravel()

    xmin = float(np.min(x_points))
    xmax = float(np.max(x_points))
    colors = [
        ROOT.kAzure + 2,
        ROOT.kOrange + 7,
        ROOT.kGreen + 2,
        ROOT.kMagenta + 1,
        ROOT.kRed + 1,
        ROOT.kCyan + 2,
        ROOT.kViolet + 6,
        ROOT.kSpring + 5,
        ROOT.kPink + 7,
        ROOT.kBlue + 3,
    ]
    keep = []

    c.cd(1)
    ROOT.gPad.SetGridx(True)
    ROOT.gPad.SetGridy(True)
    max_dev = float(np.nanmax(np.abs(uncorr_ratios - 1.0)))
    ymin = 1.0 - 1.18 * max_dev
    ymax = 1.0 + 1.18 * max_dev
    if ymin == ymax:
        ymin, ymax = 0.9, 1.1
    frame_top = ROOT.gPad.DrawFrame(xmin, ymin, xmax, ymax)
    frame_top.SetTitle("GP uncorr. / exp stat #oplus syst uncorr. for each ell")
    frame_top.GetXaxis().SetTitle("p_{T} [GeV/c]")
    frame_top.GetYaxis().SetTitle("ratio")
    keep.append(frame_top)

    one = ROOT.TLine(xmin, 1.0, xmax, 1.0)
    one.SetLineStyle(2)
    one.SetLineColor(ROOT.kGray + 2)
    one.SetLineWidth(1)
    one.Draw("SAME")
    keep.append(one)

    leg_top = ROOT.TLegend(0.72, 0.12, 0.90, 0.48)
    leg_top.SetBorderSize(0)
    leg_top.SetFillStyle(0)
    leg_top.SetNColumns(2)
    keep.append(leg_top)
    for i_ell, ell in enumerate(ell_values):
        g = ROOT.TGraph(len(x_points), x_points, uncorr_ratios[i_ell].astype(np.float64))
        g.SetName(f"g_uncorr_ratio_ell_{i_ell}_{uid}")
        g.SetLineColor(colors[i_ell % len(colors)])
        g.SetMarkerColor(colors[i_ell % len(colors)])
        g.SetLineWidth(2)
        g.SetMarkerStyle(20 + (i_ell % 10))
        g.SetMarkerSize(0.45)
        g.Draw("LP SAME")
        leg_top.AddEntry(g, f"ell={ell:.1f}", "lp")
        keep.append(g)
    leg_top.Draw()

    c.cd(2)
    ROOT.gPad.SetGridx(True)
    ROOT.gPad.SetGridy(True)
    denom = stat_exp**2 + syst_uncorr_exp**2
    frac_stat = np.divide(stat_exp**2, denom, out=np.zeros_like(stat_exp), where=denom > 0)
    frac_syst_uncorr = np.divide(syst_uncorr_exp**2, denom, out=np.zeros_like(syst_uncorr_exp), where=denom > 0)

    frame_bottom = ROOT.gPad.DrawFrame(xmin, 0.0, xmax, 1.0)
    frame_bottom.SetTitle("Experimental composition of uncorrelated variance")
    frame_bottom.GetXaxis().SetTitle("p_{T} [GeV/c]")
    frame_bottom.GetYaxis().SetTitle("fraction of stat^{2} + syst_{uncorr}^{2}")
    keep.append(frame_bottom)

    g_stat = _make_filled_band_graph(
        x_points, np.zeros_like(frac_stat), frac_stat,
        f"g_frac_stat_{uid}", ROOT.kOrange + 7, 0.55,
    )
    g_syst_uncorr = _make_filled_band_graph(
        x_points, frac_stat, frac_stat + frac_syst_uncorr,
        f"g_frac_syst_uncorr_{uid}", ROOT.kOrange - 2, 0.45,
    )
    g_stat.Draw("F SAME")
    g_syst_uncorr.Draw("F SAME")
    keep.extend([g_stat, g_syst_uncorr])

    leg_bottom = ROOT.TLegend(0.58, 0.72, 0.90, 0.88)
    leg_bottom.SetBorderSize(0)
    leg_bottom.SetFillStyle(0)
    leg_bottom.AddEntry(g_stat, "stat^{2} fraction", "f")
    leg_bottom.AddEntry(g_syst_uncorr, "syst uncorr.^{2} fraction", "f")
    leg_bottom.Draw()
    keep.append(leg_bottom)

    c._keep = keep
    c.Modified()
    c.Update()

    if hasattr(ROOT, "JSROOT") and hasattr(ROOT.JSROOT, "Draw"):
        return ROOT.JSROOT.Draw(c)
    return c


c_uncorr_ratio_composition = plot_uncorr_ratio_and_exp_composition_root_inline(
    pT,
    pointwise_ratio_scan,
    stat_exp,
    syst_uncorr_exp,
)
display(c_uncorr_ratio_composition)
